# PMC3ILV exercise: finding pathogenic repeat expansions in PureTarget long-read data

**Course:** Precision Medicine – Clinical Bioinformatics (PMC3ILV), IMC Krems, WS 2026/27
**Date of analysis:** 13 November 2026

This notebook holds the full analysis. Every command was run on a Linux server inside the conda environment `trgt-course`, from the working folder `~/PMC3ILV_exercise_data`. The notebook was executed top to bottom with `jupyter nbconvert --execute`, so all outputs below were produced by the code shown.

In [ ]:
import os
os.chdir(os.path.expanduser("~/PMC3ILV_exercise_data"))
print(os.getcwd())

---
# Part B · Set up the environment and get the data

## B1. Conda environment

The environment was created once, from a terminal (not re-run here). `--override-channels` was needed because the server's conda configuration lists `nodefaults` as a channel, which made `conda create` fail with an HTTP error.

```bash
conda create -n trgt-course --override-channels -c conda-forge -c bioconda \
    trgt stranger samtools bcftools -y
conda activate trgt-course
```

Tool versions:

In [ ]:
%%bash
echo "trgt:     $(trgt --version)"
echo "stranger: $(stranger --version)"
echo "samtools: $(samtools --version | head -1)"
echo "bcftools: $(bcftools --version | head -1)"
echo "conda env: $CONDA_DEFAULT_ENV"

## B2. Data

Data were taken from the IMC server (`/data/PMC3ILV_data`), with these commands (run once in a terminal):

```bash
cd ~
unzip /data/PMC3ILV_data/PMC3ILV_exercise_data.zip
cd PMC3ILV_exercise_data
mkdir ref
REF=/data/PMC3ILV_data/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna
ln -s $REF ref/hg38.fa
samtools faidx ref/hg38.fa
mkdir -p results
```

- **Reads:** PacBio PureTarget repeat expansion panel, 11 Coriell cell-line samples, Revio, aligned to GRCh38 (course archive `PMC3ILV_exercise_data.zip`).
- **Reference:** GRCh38 no-alt analysis set, `GCA_000001405.15_GRCh38_no_alt_analysis_set.fna` (NCBI: https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/000/001/405/GCA_000001405.15_GRCh38/seqs_for_alignment_pipelines.ucsc_ids/).

Checks:

In [ ]:
%%bash
cat README.txt
echo "---- samples.tsv"
cat samples.tsv
echo "---- number of BAM files:"
ls bam/*.bam | wc -l
ls bam/
echo "---- reference:"
ls -l ref/
head -3 ref/hg38.fa.fai

## B3. Repeat catalogue

The TRGT README says its bundled catalogue (`pathogenic_repeats.hg38.bed`) is deprecated and recommends **STRchive** (Hiatt et al., *Genome Medicine* 2025; https://strchive.org), a curated catalogue of known pathogenic repeat loci. The hg38 TRGT-format file was downloaded from the STRchive GitHub repository (branch `main`, downloaded 13 November 2026):

In [ ]:
%%bash
URL=https://raw.githubusercontent.com/dashnowlab/STRchive/main/data/catalogs/STRchive-disease-loci.hg38.TRGT.bed
[ -s repeats.bed ] || wget -q -O repeats.bed $URL
echo "Source: $URL"
head -3 repeats.bed
echo "Number of repeats in catalogue:"
grep -vc '^#' repeats.bed

**Answer B3:** the recommended catalogue is **STRchive**. The hg38 TRGT version contains **82 repeat loci**.

---
# Part C · Find the expanded alleles

## C1. Genotyping with TRGT

Settings follow `docs/puretarget.md` in the TRGT repository: **`--preset targeted`** for PureTarget data, with all other parameters at their defaults. The data are one aligned BAM per sample, so `--fail-reads` isn't used. The sex chromosomes are given with **`--karyotype`** (XX/XY), taken from `samples.tsv`.

In [ ]:
%%bash
tail -n +2 samples.tsv | while read -r SAMPLE KARYO; do
  echo "== $SAMPLE ($KARYO)"
  trgt genotype \
    --preset targeted \
    --genome ref/hg38.fa \
    --repeats repeats.bed \
    --reads bam/${SAMPLE}.bam \
    --karyotype ${KARYO} \
    --output-prefix results/${SAMPLE}
done > results/trgt_genotype.log 2>&1
echo "Warnings/errors in log:"
grep -c WARN results/trgt_genotype.log
grep -ci error results/trgt_genotype.log || true
echo "Example warnings:"
grep WARN results/trgt_genotype.log | head -5

TRGT warns that it filtered a few "impure" reads per locus (reads whose repeat sequence fits the motif poorly, typically low-accuracy reads). With hundreds of reads per locus this is expected for targeted data and doesn't affect the genotypes.

Sort and index the VCFs and the spanning-read BAMs (needed for `trgt plot`), as in `docs/tutorial.md`:

In [ ]:
%%bash
for S in $(tail -n +2 samples.tsv | cut -f1); do
  bcftools sort -Oz -o results/${S}.sorted.vcf.gz results/${S}.vcf.gz 2>/dev/null
  bcftools index -f results/${S}.sorted.vcf.gz
  samtools sort -o results/${S}.spanning.sorted.bam results/${S}.spanning.bam 2>/dev/null
  samtools index results/${S}.spanning.sorted.bam
done
ls results/ | grep sorted

## C2 (optional). Why the karyotype matters

TRGT uses the karyotype to know how many copies of each sex chromosome a sample has. For an **XY** sample, repeats on chrX (and chrY) are present in **one copy (hemizygous)**, so TRGT must call a single allele instead of forcing two. Otherwise it could invent a second allele from noise or report a false heterozygous genotype. In the results this shows as `GT=1` (one allele) for the XY samples at chrX loci: sample03 at *FMR1* and sample09 at *AR*. On this panel, the chrX loci affected are:

In [ ]:
%%bash
grep '^chrX' repeats.bed | cut -f1-4 | sed 's/;MOTIFS.*//'

## C3. Annotation with Stranger

Stranger's hg38 disease-locus table was downloaded from the Stranger repository at the release matching the installed version (v0.10.2). The `--trgt` flag tells Stranger the VCF comes from TRGT. Stranger matches the TRGT repeat IDs by gene name (e.g. `HD_HTT` → `HTT`). 31 of the 82 STRchive loci have no entry in the Stranger table, so they get no status (the "No info for repeat id" warnings).

In [ ]:
%%bash
URL=https://raw.githubusercontent.com/Clinical-Genomics/stranger/v0.10.2/stranger/resources/variant_catalog_hg38.json
[ -s stranger_hg38.json ] || wget -q -O stranger_hg38.json $URL
echo "Source: $URL"
echo "Entries with PathologicStruc (TRGT-compatible):"
grep -c PathologicStruc stranger_hg38.json

for S in $(tail -n +2 samples.tsv | cut -f1); do
  stranger --trgt -f stranger_hg38.json results/${S}.sorted.vcf.gz \
      > results/${S}.stranger.vcf 2> results/${S}.stranger.log
  bgzip -f results/${S}.stranger.vcf
  bcftools index -f results/${S}.stranger.vcf.gz
done
echo "Loci without Stranger definition (per sample):"
grep -c "No info for repeat id" results/sample01.stranger.log

In [ ]:
%%bash
echo "INFO fields added by Stranger:"
bcftools view -h results/sample01.stranger.vcf.gz | grep '^##INFO' | grep -vE 'ID=(TRID|END|MOTIFS|STRUC),'
echo
echo "Status counts over all samples (records):"
for S in $(tail -n +2 samples.tsv | cut -f1); do
  bcftools query -f "%INFO/STR_STATUS\n" results/${S}.stranger.vcf.gz
done | sort | uniq -c

**Answer C3:** Stranger compares the motif counts that TRGT reports with known disease thresholds and adds, for each repeat, the **`STR_STATUS`**: `normal`, `pre_mutation` or `full_mutation`. This is its main output. It also adds the thresholds used (`STR_NORMAL_MAX`, `STR_PATHOLOGIC_MIN`), the associated **disease**, the inheritance mode, the HGNC ID and the source of the definition. In TRGT mode it splits each record into one record per allele, so each allele gets its own status.

## C4. Table of expanded alleles

In [ ]:
%%bash
OUT=results/expanded_alleles.tsv
echo -e "sample\tlocus\tstatus\tdisease\tGT\tmotif_count\tsupporting_reads\tallele_length_bp\tnormal_max\tpathologic_min" > $OUT
for S in $(tail -n +2 samples.tsv | cut -f1); do
  bcftools query \
    -i 'INFO/STR_STATUS="pre_mutation" || INFO/STR_STATUS="full_mutation"' \
    -f "${S}\t%INFO/TRID\t%INFO/STR_STATUS\t%INFO/Disease\t[%GT\t%MC\t%SD\t%AL]\t%INFO/STR_NORMAL_MAX\t%INFO/STR_PATHOLOGIC_MIN\n" \
    results/${S}.stranger.vcf.gz
done >> $OUT
column -t -s $'\t' $OUT
echo
echo "Number of non-normal alleles: $(tail -n +2 $OUT | wc -l)"
echo "Number of different loci:     $(tail -n +2 $OUT | cut -f2 | sort -u | wc -l)"

*Notes on the table:* `GT=./1` means that Stranger split the TRGT record by allele and this row is the expanded allele. `GT=1` is a hemizygous chrX call in an XY sample. sample02 (*PABPN1*, `0/1`) wasn't split because one allele is the reference: its expanded allele is the second value (13 GCN repeats, 40 bp). For multi-motif loci the motif count is given per motif (`_`-separated, in the order of `MOTIFS`). For example, *HTT* `74_8` = 74 CAG + 8 CCG.

**Answer C4:**
- **12 non-normal alleles** (11 `full_mutation`, 1 `pre_mutation`) in 11 samples, at **10 different loci**: *ATXN3* (MJD), *PABPN1* (OPMD), *FMR1* (fragile X; two samples), *ATN1* (DRPLA), *RFC1* (CANVAS), *HTT* (Huntington), *DMPK* (DM1), *AR* (SBMA), *FXN* (Friedreich ataxia; both alleles expanded in sample10) and *ATXN1* (SCA1).
- **Longest expansion:** sample10, *FXN* (GAA), with an allele length of **3,182 bp**, about **21× a 150 bp short read** (3182 / 150 ≈ 21). The next are sample03 *FMR1* (2,655 bp, about 18×) and sample05 *RFC1* (2,086 bp, about 14×). No short read can span such an allele, or even reach across it to anchor in the flanking sequence on both sides.

---
# Part D · Look at the reads: length, methylation, interruptions

Waterfall plots were made with `trgt plot --plot-type waterfall` from the TRGT (not Stranger) sorted VCF and the sorted spanning-read BAM. `--show motifs` colours the repeat by motif and `--show meth` colours CpGs by methylation (5mC probability).

In [ ]:
%%bash
mkdir -p results/plots
plot() {  # plot SAMPLE REPEAT_ID SHOW
  trgt plot \
    --genome ref/hg38.fa \
    --repeats repeats.bed \
    --vcf results/$1.sorted.vcf.gz \
    --spanning-reads results/$1.spanning.sorted.bam \
    --repeat-id $2 \
    --plot-type waterfall \
    --show $3 \
    --image results/plots/$1_$2_$3.svg
}
plot sample03 FXS_FMR1   motifs   # D1
plot sample07 DM1_DMPK   motifs   # D1
plot sample10 FRDA_FXN   motifs   # D1
plot sample03 FXS_FMR1   meth     # D2 full mutation
plot sample08 FXS_FMR1   meth     # D2 premutation
plot sample11 SCA1_ATXN1 motifs   # D4
plot sample06 HD_HTT     motifs   # D4
ls -lh results/plots/

In [ ]:
from IPython.display import SVG, display, Markdown
def show(name, title):
    display(Markdown(f"**{title}** (`results/plots/{name}.svg`)"))
    display(SVG(filename=f"results/plots/{name}.svg"))

## D1. Waterfall plots of three expanded loci

A **waterfall plot** shows every read that spans the repeat as one horizontal row, with the flanking sequence (teal) on both sides and the repeat in the middle, coloured by motif. The reads are grouped by allele and sorted by repeat length, so each allele appears as a block of rows and its length can be read off directly. Grey marks sequence that doesn't match the catalogue motifs: consistent grey columns are real interruptions, while scattered grey is sequencing noise.

In [ ]:
show("sample03_FXS_FMR1_motifs", "sample03 – FMR1 (fragile X), full mutation")

In [ ]:
show("sample07_DM1_DMPK_motifs", "sample07 – DMPK (myotonic dystrophy type 1)")

In [ ]:
show("sample10_FRDA_FXN_motifs", "sample10 – FXN (Friedreich ataxia)")

**Answer D1:**
- **sample03 FMR1:** **one allele**, because the sample is XY and *FMR1* is on chrX. It is an almost pure CGG repeat with read lengths from about 0.6 kb to about 3 kb.
- **sample07 DMPK:** **two alleles**: a short normal allele (many reads) and an expanded pure CAG allele of about 1.1–1.5 kb.
- **sample10 FXN:** **two alleles, both expanded** (about 1.5–1.9 kb and about 2.5–3.2 kb), as expected for the recessive Friedreich ataxia. The alternating GAA/AAG colours aren't interruptions: GAA and AAG are the same repeat in different frames, both listed in the catalogue.
- **No, the reads of an expanded allele don't all have the same length.** They spread over hundreds of bp up to kb, much more than the reads of normal alleles. This reflects **somatic mosaicism**, the instability of expanded repeats between cells. PureTarget uses no PCR, so each read is an original molecule and the spread is biological, not a PCR stutter artefact.

## D2. Methylation at FMR1

In [ ]:
show("sample03_FXS_FMR1_meth", "sample03 – FMR1 full mutation, methylation")

In [ ]:
show("sample08_FXS_FMR1_meth", "sample08 – FMR1 premutation, methylation")

In [ ]:
%%bash
echo -e "sample\tlocus\tGT\tmotif_count\tallele_length\tmean_methylation(AM)"
for S in sample03 sample08; do
  bcftools query -i 'INFO/TRID="FXS_FMR1"' -f "${S}\t%INFO/TRID\t[%GT\t%MC\t%AL\t%AM]\n" results/${S}.stranger.vcf.gz
done

**Answer D2:**
- **sample03 (full mutation, about 877 CGG, XY):** the expanded allele is **methylated**. Almost all reads are red (mean methylation 0.86). Fully grey reads have no methylation call (no kinetic data) and aren't unmethylated. This is the typical fragile X pattern: hypermethylation of the full mutation silences *FMR1*, which causes loss of FMRP.
- **sample08 (premutation, about 78 CGG, XX):** the two alleles are clearly different. The normal allele (about 30 CGG) is **unmethylated** and the premutation allele is **methylated** (mean 0.94). Premutations are normally *not* methylated, but this is a female sample. In females one X is methylated as part of normal X inactivation. In a (clonal) cell line, X inactivation is usually **completely skewed**, so the methylated allele here most likely reflects that the premutation sits on the inactive X, not premutation-driven silencing.

## D3. Why long reads carry methylation and short reads don't

PacBio sequences **native DNA** without amplification. During sequencing, the polymerase's kinetics (inter-pulse duration and pulse width) change at a modified base, so 5mC at CpGs is detected directly from the same read. Standard short-read libraries are **PCR-amplified**, and the copies don't keep the methylation of the template. Sequencing by synthesis also reads only the four bases. Methylation with short reads therefore needs a separate chemical or enzymatic conversion (bisulfite / EM-seq), which degrades the DNA, lowers sequence complexity and makes mapping in repeats even harder. In imprinting disorders the diagnosis depends on parent-of-origin methylation, so long reads can give the sequence variant, the haplotype (phase) and the methylation of each allele in one test.

## D4. Interruptions in ATXN1 and HTT

In [ ]:
show("sample11_SCA1_ATXN1_motifs", "sample11 – ATXN1 (SCA1)")

In [ ]:
show("sample06_HD_HTT_motifs", "sample06 – HTT (Huntington disease)")

**Answer D4:**
- **ATXN1 (sample11):** the **normal allele** (about 30 repeats, about 148 bp) is **not pure**. Every read has two grey columns in the middle of the repeat: the **CAT interruptions** typical of normal *ATXN1* alleles. The **expanded allele** (43 repeats) is a **pure, uninterrupted** CTG/CAG run.
- **HTT (sample06):** in **both alleles** the CAG run is followed, at the same position in every read, by a short grey stretch and then the CCG repeat (purple). This is the normal *HTT* structure: (CAG)n–**CAA**–CAG–**CCA**–(CCG)n. Grey = the CAA and CCA interruptions, which aren't in the catalogue motif list. The expanded allele has a pure run of 74 CAG before the CAA interruption. Neither allele shows loss or duplication of the CAA interruption.

## D5 (optional). What interruptions mean
- **ATXN1:** CAT interruptions stabilise the repeat and are protective. Alleles in the expanded size range that still carry CAT interruptions usually don't cause SCA1, so it is the uninterrupted CAG length that matters for diagnosis.
- **HTT:** loss of the CAA interruption (pure CAG directly followed by CCG) is linked to **earlier** age at onset than predicted from the CAG count, and a duplicated CAA–CAG to later onset. The pure CAG length predicts onset better than the total repeat length (GeM-HD Consortium, *Cell* 2019).
- **FMR1:** AGG interruptions within the CGG repeat reduce the risk that a maternal premutation expands to a full mutation in the next generation.

---
# Reproducibility summary

| Item | Value |
|---|---|
| Server | IMC teaching server, conda env `trgt-course` (channels conda-forge, bioconda) |
| Tools | trgt 5.1.0, stranger 0.10.2, samtools 1.25, bcftools 1.25 |
| Reads | `PMC3ILV_exercise_data.zip` (course data, `/data/PMC3ILV_data`): PacBio PureTarget, 11 Coriell samples, Revio, GRCh38-aligned |
| Reference | GRCh38 no-alt analysis set `GCA_000001405.15_GRCh38_no_alt_analysis_set.fna` (NCBI GenBank FTP) |
| Repeat catalogue | STRchive `STRchive-disease-loci.hg38.TRGT.bed` (github.com/dashnowlab/STRchive, `main`, downloaded 2026-11-13), 82 loci |
| Stranger table | `variant_catalog_hg38.json` from github.com/Clinical-Genomics/stranger, tag v0.10.2 |
| TRGT genotype | `--preset targeted`, `--karyotype` XX/XY from `samples.tsv`, other parameters default |
| Stranger | `--trgt -f stranger_hg38.json` |
| TRGT plot | `--plot-type waterfall`, `--show motifs` or `--show meth` |
